In [0]:
import pandas as pd

In [0]:
local_excel = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/NBA Model Architecture.xlsx"

In [0]:
import mlflow

RUN_ID = "aac10414faaf4438b51669de346fe7d9"

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact(
        local_excel,
        artifact_path="config"
    )

print("Done")

In [0]:
digital_in_top_1k_df=pd.read_csv("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/p12.csv")

In [0]:
digital_in_top_1k_df.head()

In [0]:
import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from pyspark.sql import SparkSession

# ============================================================
# Update these
# ============================================================
RUN_ID = "76f91e2d55e448c98ec60d84b87bf6bc"

TRAIN_PRED_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/train_4c_76f91e2d.parquet"
TEST_PRED_PATH  = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/test_4c_76f91e2d.parquet"

ARTIFACT_FOLDER = "lift_charts"
run_id_short = RUN_ID[:8]

# ============================================================
# Load parquet files
# ============================================================
def load_predictions(parquet_path: str) -> pd.DataFrame:
    try:
        df = pd.read_parquet(parquet_path)
    except Exception:
        spark = SparkSession.builder.getOrCreate()
        df = spark.read.parquet(parquet_path).toPandas()

    rename_map = {}
    for col in df.columns:
        lc = col.lower()
        if lc == "customer_id":
            rename_map[col] = "CUSTOMER_ID"
        elif lc == "actual":
            rename_map[col] = "Actual"
        elif lc == "predicted":
            rename_map[col] = "Predicted"

    if rename_map:
        df = df.rename(columns=rename_map)

    df = df[["CUSTOMER_ID", "Actual", "Predicted"]].copy()
    df["Actual"] = pd.to_numeric(df["Actual"], errors="coerce")
    df["Predicted"] = pd.to_numeric(df["Predicted"], errors="coerce")
    df = df.dropna(subset=["Actual", "Predicted"]).reset_index(drop=True)
    return df

# ============================================================
# Lift chart matching your screenshot
# ============================================================
def create_lift_chart_like_image(actual, predicted, title, save_path, n_deciles=10):
    df = pd.DataFrame({
        "Actual": np.asarray(actual).flatten(),
        "Predicted": np.asarray(predicted).flatten()
    })

    df = df.sort_values("Predicted", ascending=False).reset_index(drop=True)

    df["Decile"] = pd.qcut(df.index, q=n_deciles, labels=False, duplicates="drop") + 1
    df["Decile"] = df["Decile"] * (100 // n_deciles)

    overall_actual_mean = df["Actual"].mean()
    overall_pred_mean = df["Predicted"].mean()

    summary = df.groupby("Decile").agg(
        n=("Actual", "size"),
        actual_mean=("Actual", "mean"),
        pred_mean=("Predicted", "mean")
    ).reset_index()

    summary["actual_lift"] = summary["actual_mean"] / overall_actual_mean if overall_actual_mean > 0 else 0
    summary["predicted_lift"] = summary["pred_mean"] / overall_pred_mean if overall_pred_mean > 0 else 0

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        summary["Decile"], summary["predicted_lift"],
        marker="o", linewidth=2, label="Predicted lift"
    )

    ax.plot(
        summary["Decile"], summary["actual_lift"],
        marker="s", linewidth=2, linestyle="--", label="Actual lift"
    )

    ax.set_xlabel("Decile (ranked by Predicted NBRx)")
    ax.set_ylabel("Per-decile lift (mean / overall mean)")
    ax.set_title(title)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    ax.set_xticks([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return summary

# ============================================================
# Main
# ============================================================
train_df = load_predictions(TRAIN_PRED_PATH)
test_df = load_predictions(TEST_PRED_PATH)

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_id=RUN_ID):
    train_png = os.path.join(tempfile.gettempdir(), f"train_lift_chart_{run_id_short}.png")
    test_png  = os.path.join(tempfile.gettempdir(), f"test_lift_chart_{run_id_short}.png")
    train_csv = os.path.join(tempfile.gettempdir(), f"train_lift_summary_{run_id_short}.csv")
    test_csv  = os.path.join(tempfile.gettempdir(), f"test_lift_summary_{run_id_short}.csv")

    train_summary = create_lift_chart_like_image(
        actual=train_df["Actual"],
        predicted=train_df["Predicted"],
        title="Train Lift Chart - NBA Kerendia NBRx",
        save_path=train_png
    )

    test_summary = create_lift_chart_like_image(
        actual=test_df["Actual"],
        predicted=test_df["Predicted"],
        title="Test Lift Chart - NBA Kerendia NBRx",
        save_path=test_png
    )

    train_summary.to_csv(train_csv, index=False)
    test_summary.to_csv(test_csv, index=False)

    mlflow.log_artifact(train_png, artifact_path=ARTIFACT_FOLDER)
    mlflow.log_artifact(test_png, artifact_path=ARTIFACT_FOLDER)
    mlflow.log_artifact(train_csv, artifact_path=ARTIFACT_FOLDER)
    mlflow.log_artifact(test_csv, artifact_path=ARTIFACT_FOLDER)

print("Logged updated lift charts to the same run.")
print(f"{ARTIFACT_FOLDER}/train_lift_chart_76f91e2d.png")
print(f"{ARTIFACT_FOLDER}/test_lift_chart_76f91e2d.png")
print(f"{ARTIFACT_FOLDER}/train_lift_summary_76f91e2d.csv")
print(f"{ARTIFACT_FOLDER}/test_lift_summary_76f91e2d.csv")

In [0]:
import json
import mlflow

RUN_ID = "5451650042fe400b955f60ef229ede82"

df_to_log = digital_in_top_1k_df

ARTIFACT_FOLDER = "digital_in_top_1k"
CSV_NAME = "digital_in_top_1k.csv"

local_csv_path = f"/tmp/{CSV_NAME}"

# ============================================================
# Save CSV locally exactly as dataframe columns are
# ============================================================

# If pandas dataframe
try:
    df_to_log.to_csv(local_csv_path, index=False)

# If Spark dataframe
except AttributeError:
    (
        df_to_log
        .toPandas()
        .to_csv(local_csv_path, index=False)
    )

print("Local CSV created:")
print(local_csv_path)

# ============================================================
# Optional: delete old run_config artifact folder
# This deletes only run_config, not the full MLflow run.
# ============================================================

client = mlflow.tracking.MlflowClient()
run = client.get_run(RUN_ID)

artifact_uri = run.info.artifact_uri

old_run_config_path = artifact_uri.rstrip("/") + "/run_config"

print("Deleting old run_config folder:")
print(old_run_config_path)

try:
    dbutils.fs.rm(old_run_config_path, True)
    print("Old run_config folder deleted.")
except Exception as e:
    print("Could not delete old run_config folder, continuing.")
    print(e)

# ============================================================
# Optional: delete old digital_in_top_1k file if already present
# ============================================================

target_csv_path = artifact_uri.rstrip("/") + f"/{ARTIFACT_FOLDER}/{CSV_NAME}"

print("Deleting old target CSV if exists:")
print(target_csv_path)

try:
    dbutils.fs.rm(target_csv_path, True)
    print("Old digital_in_top_1k CSV deleted.")
except Exception as e:
    print("No existing target CSV found, continuing.")
    print(e)

# ============================================================
# Log CSV to same existing MLflow run
# Folder name: digital_in_top_1k
# File name  : digital_in_top_1k.csv
# ============================================================

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact(
        local_csv_path,
        artifact_path=ARTIFACT_FOLDER
    )

print("Logged CSV successfully.")
print("Artifact folder:", ARTIFACT_FOLDER)
print("CSV name:", CSV_NAME)
print("Refresh MLflow UI.")